# 推荐日志：检查与清洗

对三张推荐行为日志执行结构、重复、时长、时间字段与跨表参照完整性检查，并分别生成标准推荐清洗表与随机推荐清洗表。

- `df`：随机推荐日志（2022-04-22—2022-05-08）
- `df1`：标准推荐日志（2022-04-08—2022-04-21）
- `df2`：标准推荐日志（2022-04-22—2022-05-08）


In [1]:
from pathlib import Path

import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    """从当前目录向上定位作品集根目录。"""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Python").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("未找到项目根目录，请从 KuaiRand_Pure 目录或其子目录运行。")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MYSQL_IMPORT_DIR = PROJECT_ROOT / "data" / "mysql_import"


In [2]:
#导入log_standard_4_22_to_5_08
df2=pd.read_csv(RAW_DIR / "log_standard_4_22_to_5_08_pure.csv")


# 表df2的数据质量检测


In [3]:
#检查表格的基本结构
print("行数和列数：", df2.shape)
print("索引名:",df2.index)
print("列名：", df2.columns.tolist())
df2.info()
df2.tail(10)

行数和列数： (295497, 19)
索引名: RangeIndex(start=0, stop=295497, step=1)
列名： ['user_id', 'video_id', 'date', 'hourmin', 'time_ms', 'is_click', 'is_like', 'is_follow', 'is_comment', 'is_forward', 'is_hate', 'long_view', 'play_time_ms', 'duration_ms', 'profile_stay_time', 'comment_stay_time', 'is_profile_enter', 'is_rand', 'tab']
<class 'pandas.DataFrame'>
RangeIndex: 295497 entries, 0 to 295496
Data columns (total 19 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   user_id            295497 non-null  int64
 1   video_id           295497 non-null  int64
 2   date               295497 non-null  int64
 3   hourmin            295497 non-null  int64
 4   time_ms            295497 non-null  int64
 5   is_click           295497 non-null  int64
 6   is_like            295497 non-null  int64
 7   is_follow          295497 non-null  int64
 8   is_comment         295497 non-null  int64
 9   is_forward         295497 non-null  int64
 10  is_hate 

,user_id,video_id,date,hourmin,time_ms,is_click,is_like,is_follow,is_comment,is_forward,is_hate,long_view,play_time_ms,duration_ms,profile_stay_time,comment_stay_time,is_profile_enter,is_rand,tab
295487,27283,6975,20220505,2300,1651763352355,1,0,0,0,0,0,0,7051,23933,0,0,0,0,1
295488,27283,204,20220506,1400,1651817934837,0,0,0,0,0,0,0,4562,16940,0,0,0,0,1
295489,27283,4148,20220507,1500,1651907267499,1,0,0,0,0,0,0,10209,0,0,0,0,0,1
295490,27283,1950,20220507,1500,1651907489033,1,0,0,0,0,0,0,726,11833,0,0,0,0,1
295491,27284,5769,20220430,800,1651277081623,1,0,0,0,0,0,1,34207,32233,0,0,0,0,2
295492,27284,1486,20220502,1100,1651461216855,0,0,0,0,0,0,0,1324,131066,0,0,0,0,4
295493,27284,6132,20220502,1600,1651478750711,0,0,0,0,0,0,0,1616,25100,0,0,0,0,5
295494,27284,6132,20220502,1600,1651478750711,0,0,0,0,0,0,0,1616,25100,0,0,0,0,5
295495,27284,3259,20220507,1800,1651918814747,1,0,0,0,0,0,1,49400,48133,0,0,0,0,2
295496,27284,1694,20220508,700,1651964016379,1,0,0,0,0,0,1,44010,44660,0,0,0,0,5


In [4]:
#检查缺失值
print("各列缺失值数量：")
print(df2.isna().sum())

#检查重复值
print("完全重复的行数：", df2.duplicated().sum())

dup_all = df2[df2.duplicated(keep=False)].sort_values(
    ["user_id", "video_id", "time_ms"]
)

print("参与完全重复的总行数：", dup_all.shape[0])
print("完全重复的行数比例：", df2.duplicated().mean())

dup_all.head(20)



各列缺失值数量：
user_id              0
video_id             0
date                 0
hourmin              0
time_ms              0
is_click             0
is_like              0
is_follow            0
is_comment           0
is_forward           0
is_hate              0
long_view            0
play_time_ms         0
duration_ms          0
profile_stay_time    0
comment_stay_time    0
is_profile_enter     0
is_rand              0
tab                  0
dtype: int64
完全重复的行数： 6378
参与完全重复的总行数： 12743
完全重复的行数比例： 0.0215839754718322


,user_id,video_id,date,hourmin,time_ms,is_click,is_like,is_follow,is_comment,is_forward,is_hate,long_view,play_time_ms,duration_ms,profile_stay_time,comment_stay_time,is_profile_enter,is_rand,tab
152766,3,2684,20220501,1100,1651374316788,1,0,0,0,0,0,1,100072,317633,0,0,0,0,6
152767,3,2684,20220501,1100,1651374316788,1,0,0,0,0,0,1,100072,317633,0,0,0,0,6
31,7,5474,20220423,2200,1650722026682,0,0,0,0,0,0,0,2061,1177720,0,0,0,0,6
32,7,5474,20220423,2200,1650722026682,0,0,0,0,0,0,0,2061,1177720,0,0,0,0,6
67,16,5975,20220427,2200,1651068654698,0,0,0,0,0,0,0,4165,21021,0,0,0,0,6
68,16,5975,20220427,2200,1651068654698,0,0,0,0,0,0,0,4165,21021,0,0,0,0,6
93,23,5928,20220428,2300,1651159091290,0,0,0,0,0,0,0,1489,7450,0,0,0,0,6
94,23,5928,20220428,2300,1651159091290,0,0,0,0,0,0,0,1489,7450,0,0,0,0,6
131,29,5975,20220426,1900,1650973033295,0,0,0,0,0,0,0,1182,21021,0,0,0,0,6
132,29,5975,20220426,1900,1650973033295,0,0,0,0,0,0,0,1182,21021,0,0,0,0,6


In [5]:
#检查单个字段的有效性
#检查二值0，1字段

binary_cols = [
    "is_click",
    "is_like",
    "is_follow",
    "is_comment",
    "is_forward",
    "is_hate",
    "long_view",
    "is_profile_enter",
    "is_rand"
]
print(df2[binary_cols])
df2[binary_cols].agg(["min", "max", "nunique"])
#DataFrame.agg([函数1, 函数2, 函数3,...])


        is_click  is_like  is_follow  is_comment  is_forward  is_hate  \
0              0        0          0           0           0        0   
1              0        0          0           0           0        0   
2              0        0          0           0           0        0   
3              0        0          0           0           0        0   
4              0        0          0           0           0        0   
...          ...      ...        ...         ...         ...      ...   
295492         0        0          0           0           0        0   
295493         0        0          0           0           0        0   
295494         0        0          0           0           0        0   
295495         1        0          0           0           0        0   
295496         1        0          0           0           0        0   

        long_view  is_profile_enter  is_rand  
0               0                 0        0  
1               0            

,is_click,is_like,is_follow,is_comment,is_forward,is_hate,long_view,is_profile_enter,is_rand
min,0,0,0,0,0,0,0,0,0
max,1,1,1,1,1,1,1,1,0
nunique,2,2,2,2,2,2,2,2,1


In [6]:
print("最早日期：", df2["date"].min())
print("最晚日期：", df2["date"].max())

print("hourmin的所有取值：")
print(sorted(df2["hourmin"].unique()))

print("tab的取值数量：")
print(df2["tab"].value_counts().sort_index())


最早日期： 20220422
最晚日期： 20220508
hourmin的所有取值：
[np.int64(0), np.int64(100), np.int64(200), np.int64(300), np.int64(400), np.int64(500), np.int64(600), np.int64(700), np.int64(800), np.int64(900), np.int64(1000), np.int64(1100), np.int64(1200), np.int64(1300), np.int64(1400), np.int64(1500), np.int64(1600), np.int64(1700), np.int64(1800), np.int64(1900), np.int64(2000), np.int64(2100), np.int64(2200), np.int64(2300)]
tab的取值数量：
tab
0      28074
1     223513
2       9073
3        440
4      17641
5        767
6      13114
7        171
8       1358
9        192
10        72
11       308
12       638
13       116
14        20
Name: count, dtype: int64


In [7]:
#排查业务逻辑不合理的地方
print("视频时长小于0的行数：", (df2["duration_ms"] < 0).sum())
print("视频时长等于0的行数：", (df2["duration_ms"] == 0).sum())
print("播放时长小于0的行数：", (df2["play_time_ms"] < 0).sum())
print("播放时长等于0的行数：", (df2["play_time_ms"] == 0).sum())
print("播放时长大于视频时长的行数：",
      (df2["play_time_ms"] > df2["duration_ms"]).sum())


视频时长小于0的行数： 0
视频时长等于0的行数： 4798
播放时长小于0的行数： 0
播放时长等于0的行数： 30509
播放时长大于视频时长的行数： 44874


## 表df2的数据质量检查结论：

1. df2共有295497行、19列，无显式缺失值。
2. 发现6378条完全重复记录，占比约2.16%。
3. 重复记录的全部字段一致，包括用户、视频和毫秒级时间戳，
   判断为重复日志，后续删除重复副本，仅保留第一次出现的记录。
4. duration_ms小于等于0、播放时长为0及播放时长超过视频时长的记录，
   暂时保留，进一步调查业务含义。


## 清洗数据

In [8]:
# 1.删除重复记录

rows_before = df2.shape[0]
df2 = df2.drop_duplicates().reset_index(drop=True)
#reset_index(drop=True)重新生成索引，并丢弃旧索引。

rows_after = df2.shape[0]

print("去重前行数：", rows_before)
print("去重后行数：", rows_after)
print("删除行数：", rows_before - rows_after)
print("去重后重复行数：", df2.duplicated().sum())


去重前行数： 295497
去重后行数： 289119
删除行数： 6378
去重后重复行数： 0


In [9]:
# 2.调查去重后，视频时长小于等于0的异常记录。
bad_duration = df2[df2["duration_ms"] <= 0]
print("去重后异常行数：", bad_duration.shape[0])

print("异常时长有哪些取值：")
print(bad_duration["duration_ms"].value_counts().sort_index())

print("涉及的视频数量：", bad_duration["video_id"].nunique())

print("出现次数最多的异常视频：")
print(bad_duration["video_id"].value_counts().head(10))

print("异常记录中，播放时长大于0的行数：",
      (bad_duration["play_time_ms"] > 0).sum())

print('异常记录中，播放时长大于0的视频数量',
      bad_duration[bad_duration['play_time_ms']>0]['video_id'].nunique()
                                                          )
bad_duration.head(10)


去重后异常行数： 4798
异常时长有哪些取值：
duration_ms
0    4798
Name: count, dtype: int64
涉及的视频数量： 190
出现次数最多的异常视频：
video_id
2594    663
5613    320
2054    215
2360    182
748     180
6692    171
4750    144
6280    117
6328    115
473     115
Name: count, dtype: int64
异常记录中，播放时长大于0的行数： 3962
异常记录中，播放时长大于0的视频数量 168


,user_id,video_id,date,hourmin,time_ms,is_click,is_like,is_follow,is_comment,is_forward,is_hate,long_view,play_time_ms,duration_ms,profile_stay_time,comment_stay_time,is_profile_enter,is_rand,tab
6,1,6328,20220429,2100,1651238509407,0,0,0,0,0,0,0,5584,0,0,0,0,0,1
42,10,6280,20220501,800,1651362785665,0,0,0,0,0,0,0,1623,0,0,0,0,0,1
72,16,4771,20220505,1400,1651730252788,1,0,0,0,0,0,0,10099,0,0,952,0,0,0
74,17,5135,20220423,2200,1650722762966,0,0,0,0,0,0,0,4703,0,0,0,0,0,1
130,29,2594,20220428,2000,1651148033856,0,0,0,0,0,0,0,3167,0,0,0,0,0,1
146,33,2594,20220430,900,1651280495095,0,0,0,0,0,0,0,1424,0,0,0,0,0,1
162,38,2594,20220426,1200,1650947852892,0,0,0,0,0,0,0,2486,0,0,0,0,0,1
231,53,4771,20220506,200,1651776334792,0,0,0,0,0,0,0,1463,0,0,0,0,0,1
270,62,7474,20220423,1400,1650693389724,0,0,0,0,0,0,0,877,0,0,0,0,0,1
320,65,1566,20220430,1900,1651316211514,0,0,0,0,0,0,0,1634,0,0,0,0,0,1


检查结论（视频时长异常）

在去除df2表中的重复行后得到：
- 异常行数4798
- 异常视频数 190
- 播放时长大于0的异常行数 3962
- 播放时长大于0的异常视频数 168

对 df2 的异常值按“调查—验证补充来源—修改主表—复查结果”的顺序处理。


In [10]:
#导入video_features_basic
video_features_basic = pd.read_csv(RAW_DIR / "video_features_basic_pure.csv")
print(video_features_basic.info())
print(video_features_basic.shape)
video_features_basic.head()


<class 'pandas.DataFrame'>
RangeIndex: 7583 entries, 0 to 7582
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   video_id        7583 non-null   int64  
 1   author_id       7583 non-null   int64  
 2   video_type      7583 non-null   str    
 3   upload_dt       7583 non-null   str    
 4   upload_type     7583 non-null   str    
 5   visible_status  7583 non-null   float64
 6   video_duration  7344 non-null   float64
 7   server_width    7583 non-null   float64
 8   server_height   7583 non-null   float64
 9   music_id        7583 non-null   int64  
 10  music_type      7380 non-null   float64
 11  tag             7487 non-null   str    
dtypes: float64(5), int64(3), str(4)
memory usage: 902.4 KB
None
(7583, 12)


,video_id,author_id,video_type,upload_dt,upload_type,visible_status,video_duration,server_width,server_height,music_id,music_type,tag
0,0,7349781,NORMAL,2022-04-10,LongImport,0.0,87433.0,720.0,1280.0,9155697141,9.0,39
1,1,2103883,NORMAL,2022-04-10,Kmovie,0.0,218066.0,720.0,1280.0,6355810746,9.0,2
2,2,5067285,NORMAL,2022-04-09,ShortImport,0.0,9233.0,720.0,1280.0,6618412736,4.0,1
3,3,7048760,NORMAL,2022-04-11,Web,0.0,16433.0,720.0,1280.0,9161677205,9.0,7
4,4,8635271,NORMAL,2022-04-09,Web,0.0,38766.0,720.0,1280.0,9141092381,9.0,9


In [11]:
#先检查视频基础表的 video_id 是否唯一
print(
    "视频基础表中重复的video_id数量：",
    video_features_basic["video_id"].duplicated().sum()
)


视频基础表中重复的video_id数量： 0


In [12]:
#只取190个异常视频，不合并全部4798行
bad_video = bad_duration[
    ["video_id", "duration_ms"]
].drop_duplicates()

print("异常视频表合并前行数：", bad_video.shape[0])


异常视频表合并前行数： 190


In [13]:
#开始合并，合并后会生成一张新的表
bad_video_check = pd.merge(
    bad_video,
    video_features_basic[["video_id", "video_duration"]],
    on="video_id",
    how="left"
)
bad_video_check.tail(10)


,video_id,duration_ms,video_duration
180,5902,0,NaN
181,2493,0,NaN
182,1012,0,NaN
183,5358,0,NaN
184,6480,0,NaN
185,2593,0,NaN
186,578,0,NaN
187,5812,0,NaN
188,4306,0,NaN
189,6185,0,NaN


In [14]:
#检查合并结果，判断补充来源是否可靠且有效。
print("合并后行数：", bad_video_check.shape[0])

print(
    "基础表中仍然缺少视频时长的数量：",
    bad_video_check["video_duration"].isna().sum()
)

print(
    "基础表中可以补充时长的视频数量：",
    (bad_video_check["video_duration"] > 0).sum()
)


合并后行数： 190
基础表中仍然缺少视频时长的数量： 190
基础表中可以补充时长的视频数量： 0


### 检查结论（视频时长异常）

合并的调查结果是：**完全无效**。
因此针对df2表中的duration_ms异常的数据，我们无法**补值**。

进一步定位“完全无效”的原因：
1. 基础表中存在这个 video_id，但它的 video_duration 本身是空值；
2. 基础表中根本找不到这个 video_id，所以合并不到。


In [15]:
print(
    "基础表中完全找不到的异常视频数量：",
    (~bad_video["video_id"].isin(
        video_features_basic["video_id"]
    )).sum()
)
basic_missing_duration = video_features_basic[
    video_features_basic["video_duration"].isna()
]

print(
    "视频基础表本身缺少时长的视频数量：",
    basic_missing_duration.shape[0]
)

print(
    "190个异常视频中，基础表存在但时长为空的数量：",
    bad_video["video_id"].isin(
        basic_missing_duration["video_id"]
    ).sum()
)


基础表中完全找不到的异常视频数量： 0
视频基础表本身缺少时长的视频数量： 239
190个异常视频中，基础表存在但时长为空的数量： 190


### 检查结论（视频时长异常）
   发现4798条duration_ms为0的记录，涉及190个视频。
   190个视频均能在视频基础表中找到，但video_duration同样为空，
   判断为数据源本身缺少视频时长。

   保留相关行为记录和原始duration_ms字段，增加**时长缺失标记**，
   依赖视频时长的指标计算时排除这些记录。


In [16]:
# 标记视频时长缺失
df2["is_duration_missing"] = (
    df2["duration_ms"] <= 0
).astype("int8")

# 保留正常时长，把0转换为NaN
df2["duration_ms_clean"] = df2["duration_ms"].where(
    df2["duration_ms"] > 0
)


In [17]:
print("时长缺失标记分布：")
print(df2["is_duration_missing"].value_counts().sort_index())

print("新增字段的数据类型：")
print(
    df2[
        ["duration_ms", "duration_ms_clean", "is_duration_missing"]
    ].dtypes
)

df2.loc[
    df2["is_duration_missing"] == 1,
    [
        "video_id",
        "play_time_ms",
        "duration_ms",
        "duration_ms_clean",
        "is_duration_missing"
    ]
].head(10)


时长缺失标记分布：
is_duration_missing
0    284321
1      4798
Name: count, dtype: int64
新增字段的数据类型：
duration_ms              int64
duration_ms_clean      float64
is_duration_missing       int8
dtype: object


,video_id,play_time_ms,duration_ms,duration_ms_clean,is_duration_missing
6,6328,5584,0,NaN,1
42,6280,1623,0,NaN,1
72,4771,10099,0,NaN,1
74,5135,4703,0,NaN,1
130,2594,3167,0,NaN,1
146,2594,1424,0,NaN,1
162,2594,2486,0,NaN,1
231,4771,1463,0,NaN,1
270,7474,877,0,NaN,1
320,1566,1634,0,NaN,1


### 检查结论（播放进度超过100%）


In [18]:
df2["play_ratio_raw"] = (
    df2["play_time_ms"] / df2["duration_ms_clean"]
)


In [19]:
print("播放进度描述统计：")

print(
    df2["play_ratio_raw"].describe(
        percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
    )
)

print(
    "播放进度超过100%的行数：",
    (df2["play_ratio_raw"] > 1).sum()
)

print(
    "播放进度超过200%的行数：",
    (df2["play_ratio_raw"] > 2).sum()
)

print(
    "播放进度超过500%的行数：",
    (df2["play_ratio_raw"] > 5).sum()
)

播放进度描述统计：
count    284321.000000
mean          0.352364
std           0.601660
min           0.000000
50%           0.095467
75%           0.578702
90%           1.030873
95%           1.127204
99%           2.045082
max          68.310126
Name: play_ratio_raw, dtype: float64
播放进度超过100%的行数： 40715
播放进度超过200%的行数： 3156
播放进度超过500%的行数： 265


In [20]:
df2.nlargest(
    10,
    "play_ratio_raw"
)[
    [
        "user_id",
        "video_id",
        "play_time_ms",
        "duration_ms_clean",
        "play_ratio_raw",
        "long_view"
    ]
]


,user_id,video_id,play_time_ms,duration_ms_clean,play_ratio_raw,long_view
206968,11257,3310,418946,6133.0,68.310126,1
187962,7497,1412,517689,8233.0,62.879752,1
84094,15355,3670,416330,7071.0,58.878518,1
48433,8785,4270,329152,6933.0,47.476129,1
164739,2970,2328,389740,11300.0,34.490265,1
240215,17775,1902,360093,12266.0,29.357003,1
115813,21150,7551,195859,6750.0,29.016148,1
277338,24999,3344,410404,15033.0,27.300206,1
184768,6859,4640,898147,36766.0,24.428739,1
20117,3718,50,256644,11000.0,23.331273,1


### 检查结论（视频时长异常）
   有效视频时长记录中，约14.32%的播放进度超过100%。
   极端记录主要为短视频长时间播放，且long_view均为1，
   推测可能存在循环或重复播放，不作为脏数据删除。

   保留原始播放进度play_ratio_raw的同时，建立is_complete_play指标，没有完播 → 0; 已知已经完播 → 1; 时长缺失、无法判断 → NaN


In [21]:
df2["is_complete_play"] = (
    df2["play_time_ms"] >= df2["duration_ms_clean"]
).where(
    df2["duration_ms_clean"].notna()
).astype("Int8")



In [22]:
print(
    df2["is_complete_play"].value_counts(
        dropna=False
    ).sort_index()
)


is_complete_play
0       243593
1        40728
<NA>      4798
Name: count, dtype: Int64


**转换日期**


In [23]:
df2["date_clean"] = pd.to_datetime(
    df2["date"].astype(str),
    format="%Y%m%d"
)


In [24]:
print(df2[["date", "date_clean"]].head())
print(df2["date_clean"].dtype)


       date date_clean
0  20220422 2022-04-22
1  20220425 2022-04-25
2  20220429 2022-04-29
3  20220430 2022-04-30
4  20220502 2022-05-02
datetime64[us]


### 检查结论（验证time_ms/date/hourmin是否一致）
（转换 time_ms 是一次数据一致性验证和精细行为分析准备）


In [25]:
# 1. 将毫秒级时间戳转换为中国时间
df2["event_time"] = pd.to_datetime(
    df2["time_ms"],
    unit="ms",
    utc=True
).dt.tz_convert("Asia/Shanghai")

# 2. 根据event_time反推日期
date_from_time = (
    df2["event_time"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

# 3. 根据event_time反推小时档位
hourmin_from_time = (
    df2["event_time"].dt.hour * 100
)


# 4. 检查三个时间字段是否一致
date_mismatch = df2["date"] != date_from_time
hourmin_mismatch = df2["hourmin"] != hourmin_from_time

print(
    "日期与时间戳不一致的行数：",
    date_mismatch.sum()
)
print(
    "日期与时间戳不一致的行数占总行数的比例：",
    round(date_mismatch.sum()/df2.shape[0]*100,2),"%"
)
print(
    "小时与时间戳不一致的行数：",
    hourmin_mismatch.sum()
)
print(
    "小时与时间戳不一致的行数占总行数的比例：",
    round(hourmin_mismatch.sum()/df2.shape[0]*100,2),"%"
)

# 5. 查看转换结果
df2[
    ["date", "date_clean", "hourmin", "time_ms", "event_time"]
].head(10)


日期与时间戳不一致的行数： 2654
日期与时间戳不一致的行数占总行数的比例： 0.92 %
小时与时间戳不一致的行数： 51067
小时与时间戳不一致的行数占总行数的比例： 17.66 %


,date,date_clean,hourmin,time_ms,event_time
0,20220422,2022-04-22,1600,1650615878629,2022-04-22 16:24:38.629000+08:00
1,20220425,2022-04-25,1800,1650882384975,2022-04-25 18:26:24.975000+08:00
2,20220429,2022-04-29,2100,1651237368151,2022-04-29 21:02:48.151000+08:00
3,20220430,2022-04-30,900,1651280365845,2022-04-30 08:59:25.845000+08:00
4,20220502,2022-05-02,1100,1651461680240,2022-05-02 11:21:20.240000+08:00
5,20220428,2022-04-28,2200,1651154821828,2022-04-28 22:07:01.828000+08:00
6,20220429,2022-04-29,2100,1651238509407,2022-04-29 21:21:49.407000+08:00
7,20220430,2022-04-30,2200,1651328487418,2022-04-30 22:21:27.418000+08:00
8,20220426,2022-04-26,1700,1650965953528,2022-04-26 17:39:13.528000+08:00
9,20220427,2022-04-27,1300,1651035362303,2022-04-27 12:56:02.303000+08:00


In [26]:
df2.loc[
    date_mismatch | hourmin_mismatch,
    ["date", "hourmin", "time_ms", "event_time"]
].head(20)


,date,hourmin,time_ms,event_time
3,20220430,900,1651280365845,2022-04-30 08:59:25.845000+08:00
9,20220427,1300,1651035362303,2022-04-27 12:56:02.303000+08:00
10,20220429,1400,1651211242264,2022-04-29 13:47:22.264000+08:00
14,20220506,2100,1651841765846,2022-05-06 20:56:05.846000+08:00
17,20220507,2000,1651924316715,2022-05-07 19:51:56.715000+08:00
19,20220424,2300,1650811639885,2022-04-24 22:47:19.885000+08:00
31,20220423,2200,1650722026682,2022-04-23 21:53:46.682000+08:00
37,20220503,2100,1651582480196,2022-05-03 20:54:40.196000+08:00
40,20220422,2000,1650626943185,2022-04-22 19:29:03.185000+08:00
41,20220422,2000,1650627039892,2022-04-22 19:30:39.892000+08:00


### 检查结论（时间字段口径说明）
将 time_ms 按毫秒级 Unix 时间戳解析，并转换为 Asia/Shanghai 后，其推导出的日期、小时与原始 date、hourmin 存在不一致。

现有数据说明未明确三者的生成关系和时区口径，也没有证据支持采用固定分钟偏移进行修正，因此不对 time_ms 进行人为时间平移，也不使用其推导结果覆盖原始时间字段。

后续数据的使用说明：
- 每日趋势使用官方date转换后的date_clean；
- 小时趋势使用官方hourmin；
- 行为排序和重复判断使用time_ms；
- 不使用time_ms反推字段覆盖date或hourmin。


### 检查结论
核验推荐日志中的用户和视频是否均能在对应维度表中匹配。


In [27]:
user_features=pd.read_csv(RAW_DIR / "user_features_pure.csv")


In [28]:
#先确认基础表主键是否重复：
print(
    "用户表重复user_id数量：",
    user_features["user_id"].duplicated().sum()
)
print(
    "视频表重复video_id数量：",
    video_features_basic["video_id"].duplicated().sum()
)


用户表重复user_id数量： 0
视频表重复video_id数量： 0


In [29]:
user_not_found = ~df2["user_id"].isin(
    user_features["user_id"]
)

video_not_found = ~df2["video_id"].isin(
    video_features_basic["video_id"]
)


In [30]:
#检查无法匹配的数据
print(
    "无法匹配用户信息的日志行数：",
    user_not_found.sum()
)

print(
    "无法匹配的不同用户数量：",
    df2.loc[user_not_found, "user_id"].nunique()
)

print(
    "无法匹配视频信息的日志行数：",
    video_not_found.sum()
)

print(
    "无法匹配的不同视频数量：",
    df2.loc[video_not_found, "video_id"].nunique()
)


无法匹配用户信息的日志行数： 0
无法匹配的不同用户数量： 0
无法匹配视频信息的日志行数： 0
无法匹配的不同视频数量： 0


### 表df2的检查结论
   用户表和视频基础表的主键均无重复。
   df2中全部user_id和video_id均能在对应基础表中匹配,  跨表参照完整性正常。


---

# 表df1和表df的数据质量检测

In [31]:
#导入log_standard_4_08_to_4_21
#检查df1表的基本结构
df1=pd.read_csv(RAW_DIR / "log_standard_4_08_to_4_21_pure.csv")
df1.info()
print("数据形状：", df1.shape)
df1.head(30)


<class 'pandas.DataFrame'>
RangeIndex: 1141112 entries, 0 to 1141111
Data columns (total 19 columns):
 #   Column             Non-Null Count    Dtype
---  ------             --------------    -----
 0   user_id            1141112 non-null  int64
 1   video_id           1141112 non-null  int64
 2   date               1141112 non-null  int64
 3   hourmin            1141112 non-null  int64
 4   time_ms            1141112 non-null  int64
 5   is_click           1141112 non-null  int64
 6   is_like            1141112 non-null  int64
 7   is_follow          1141112 non-null  int64
 8   is_comment         1141112 non-null  int64
 9   is_forward         1141112 non-null  int64
 10  is_hate            1141112 non-null  int64
 11  long_view          1141112 non-null  int64
 12  play_time_ms       1141112 non-null  int64
 13  duration_ms        1141112 non-null  int64
 14  profile_stay_time  1141112 non-null  int64
 15  comment_stay_time  1141112 non-null  int64
 16  is_profile_enter   1141112 no

,user_id,video_id,date,hourmin,time_ms,is_click,is_like,is_follow,is_comment,is_forward,is_hate,long_view,play_time_ms,duration_ms,profile_stay_time,comment_stay_time,is_profile_enter,is_rand,tab
0,0,1527,20220411,1900,1649675512388,0,0,0,0,0,0,0,1385,209900,0,0,0,0,1
1,0,7405,20220416,2000,1650111976017,0,0,0,0,0,0,0,0,65400,0,0,0,0,0
2,0,6026,20220420,1600,1650444367095,0,0,0,0,0,0,0,1405,170833,0,0,0,0,1
3,1,6354,20220411,1100,1649645295928,0,0,0,0,0,0,0,0,255160,0,0,0,0,8
4,1,3645,20220411,1100,1649648827559,0,0,0,0,0,0,0,1970,79733,0,0,0,0,1
5,1,4073,20220412,300,1649706052290,1,0,0,0,0,0,1,115607,114680,0,0,0,0,1
6,1,1725,20220412,400,1649706789917,1,0,0,0,0,0,1,158156,156433,0,0,0,0,1
7,1,3891,20220412,400,1649707373426,1,0,0,0,0,0,1,62093,173800,0,0,0,0,1
8,1,5606,20220412,400,1649708149549,0,0,0,0,0,0,0,0,109320,0,0,0,0,0
9,1,4352,20220417,1100,1650165081534,1,1,0,0,0,0,1,32486,12576,0,0,0,0,1


In [32]:
#导入log_random_4_22_to_5_08
#检查df表的基本结构
df=pd.read_csv(RAW_DIR / "log_random_4_22_to_5_08_pure.csv")
df.info()
print("数据形状：", df.shape)
df.head(10)
df['date'].value_counts()


<class 'pandas.DataFrame'>
RangeIndex: 1186059 entries, 0 to 1186058
Data columns (total 19 columns):
 #   Column             Non-Null Count    Dtype
---  ------             --------------    -----
 0   user_id            1186059 non-null  int64
 1   video_id           1186059 non-null  int64
 2   date               1186059 non-null  int64
 3   hourmin            1186059 non-null  int64
 4   time_ms            1186059 non-null  int64
 5   is_click           1186059 non-null  int64
 6   is_like            1186059 non-null  int64
 7   is_follow          1186059 non-null  int64
 8   is_comment         1186059 non-null  int64
 9   is_forward         1186059 non-null  int64
 10  is_hate            1186059 non-null  int64
 11  long_view          1186059 non-null  int64
 12  play_time_ms       1186059 non-null  int64
 13  duration_ms        1186059 non-null  int64
 14  profile_stay_time  1186059 non-null  int64
 15  comment_stay_time  1186059 non-null  int64
 16  is_profile_enter   1186059 no

date
20220508    114925
20220503    109627
20220504    108543
20220507    103982
20220506    103245
20220505     97170
20220502     85874
20220430     62716
20220501     62044
20220423     52359
20220429     49595
20220424     44441
20220426     42367
20220425     41837
20220427     41737
20220428     40748
20220422     24849
Name: count, dtype: int64

In [33]:
other_logs = {
    "df": df,
    "df1": df1
}

for name, table in other_logs.items():
    print("=" * 50)
    print("表名：", name)
    print("数据形状：", table.shape)
    #检查缺失值和重复值
    print("缺失单元格总数：", table.isna().sum().sum())
    print("完全重复副本数量：", table.duplicated().sum())
    #检查单个字段的有效性
    print("最早日期：", table["date"].min())
    print("最晚日期：", table["date"].max())
    print("hourmin的所有可能取值：",table["hourmin"].unique())
    print("tab的取值数量：",table["tab"].value_counts().sort_index())
    print("视频时长小于0的行数：",(table["duration_ms"] < 0).sum())
    print("视频时长等于0的行数：",(table["duration_ms"] == 0).sum())
    print(
        "播放时长小于0的行数：",
        (table["play_time_ms"] < 0).sum()
    )
    print(
        "二值字段非法值数量：",
        (~table[binary_cols].isin([0, 1])).sum()
    )
    print(
        "无法匹配用户的日志：",
        (~table["user_id"].isin(
            user_features["user_id"]
        )).sum()
    )
    print(
        "无法匹配视频的日志：",
        (~table["video_id"].isin(
            video_features_basic["video_id"]
        )).sum()
    )


表名： df
数据形状： (1186059, 19)
缺失单元格总数： 0
完全重复副本数量： 10
最早日期： 20220422
最晚日期： 20220508
hourmin的所有可能取值： [1800 1200 1700  800  900  700 1000 1300 1500 1900 2000 2100 1400 1600
  600 1100 2300 2200    0  100  500  400  200  300]
tab的取值数量： tab
1     1178025
2         764
11       6755
14        515
Name: count, dtype: int64
视频时长小于0的行数： 0
视频时长等于0的行数： 36920
播放时长小于0的行数： 0
二值字段非法值数量： is_click            0
is_like             0
is_follow           0
is_comment          0
is_forward          0
is_hate             0
long_view           0
is_profile_enter    0
is_rand             0
dtype: int64
无法匹配用户的日志： 0
无法匹配视频的日志： 0
表名： df1
数据形状： (1141112, 19)
缺失单元格总数： 0
完全重复副本数量： 15609
最早日期： 20220409
最晚日期： 20220421
hourmin的所有可能取值： [1900 2000 1600 1100  300  400 2200  900 1000 1200 2100 1400 1500 1800
 1700 1300 2300  100  200  500  800    0  700  600]
tab的取值数量： tab
0     150013
1     834876
2      39291
3       3574
4      75524
5       3402
6      29671
7        333
8       2551
9        252
10        80
11       

### 检查结论（快速验证时长异常规则）
本节确认两张表的异常时长是否：
- 也全部为0；
- 也来自视频基础表中时长缺失的视频。


In [34]:
for name, table in other_logs.items():
    bad = table[table["duration_ms"] <= 0]

    bad_video_ids = (
        bad["video_id"]
        .drop_duplicates()
    )

    not_in_basic_missing = (
        ~bad_video_ids.isin(
            basic_missing_duration["video_id"]
        )
    ).sum()

    print("=" * 50)
    print("表名：", name)

    print("异常时长的取值：")
    print(
        bad["duration_ms"]
        .value_counts()
        .sort_index()
    )

    print(
        "涉及异常视频数量：",
        bad_video_ids.shape[0]
    )

    print(
        "基础表时长并非缺失的异常视频数量：",
        not_in_basic_missing
    )


表名： df
异常时长的取值：
duration_ms
0    36920
Name: count, dtype: int64
涉及异常视频数量： 239
基础表时长并非缺失的异常视频数量： 0
表名： df1
异常时长的取值：
duration_ms
0    24076
Name: count, dtype: int64
涉及异常视频数量： 237
基础表时长并非缺失的异常视频数量： 0


## 开始清洗 表df和表df1

In [35]:
def clean_log(table):

    rows_before = table.shape[0]

    # 1. 删除完全重复记录，并重置索引
    table = table.drop_duplicates().reset_index(drop=True)
    # 2. 创建视频时长缺失标记
    table["is_duration_missing"] = (table["duration_ms"] <= 0).astype("int8")

    # 3. 创建清洗后的视频时长
    table["duration_ms_clean"] = table["duration_ms"].where(table["duration_ms"] > 0)

    # 4. 创建原始播放进度"
    table["play_ratio_raw"] =table["play_time_ms"]/table["duration_ms_clean"]

    # 5. 创建完整播放标记，时长缺失时应为<NA>
    table["is_complete_play"] =(table["play_time_ms"]>=table["duration_ms_clean"]).where(table["duration_ms_clean"].notna()).astype("Int8")

    # 6. 将date转换为日期类型
    table["date_clean"] = pd.to_datetime(table["date"].astype(str),format="%Y%m%d")

    rows_after = table.shape[0]

    print("清洗前行数：", rows_before)
    print("清洗后行数：", rows_after)
    print("删除重复行数：", rows_before - rows_after)

    return table


In [36]:
df2.drop(
    columns=["event_time"],
    inplace=True,
    errors="ignore"
)


In [37]:
df = clean_log(df)
df1 = clean_log(df1)


清洗前行数： 1186059
清洗后行数： 1186049
删除重复行数： 10
清洗前行数： 1141112
清洗后行数： 1125503
删除重复行数： 15609


## 截止到目前，已完成三张表的数据质量检查和清洗

---

### 下面分别导出“标准推荐日志清洗表”和“随机推荐日志清洗表"

In [38]:
print("df与df1字段一致：", df.columns.equals(df1.columns))
print("df1与df2字段一致：", df1.columns.equals(df2.columns))
#字段名完全一样 + 列的顺序也必须一模一样，才返回 True。

print(
    "三张表清洗后总行数：",
    df.shape[0] + df1.shape[0] + df2.shape[0]
)


df与df1字段一致： True
df1与df2字段一致： True
三张表清洗后总行数： 2600671


**合并df1表和df2表**


In [39]:
df1["log_source"] = "standard_0408_0421"
df2["log_source"] = "standard_0422_0508"
df["log_source"] = "random_0422_0508"


In [40]:
df_standard = pd.concat(
    [df1, df2],
    ignore_index=True
)


In [41]:
print("标准推荐主表形状：", df_standard.shape)

print("来源分布：")
print(df_standard["log_source"].value_counts())

print("日期范围：")
print(
    df_standard["date_clean"].min(),
    df_standard["date_clean"].max()
)

print("is_rand分布：")
print(
    df_standard["is_rand"].value_counts(
        dropna=False
    )
)

print("随机推荐表is_rand分布：")
print(
    df["is_rand"].value_counts(
        dropna=False
    )
)


标准推荐主表形状： (1414622, 25)
来源分布：
log_source
standard_0408_0421    1125503
standard_0422_0508     289119
Name: count, dtype: int64
日期范围：
2022-04-09 00:00:00 2022-05-08 00:00:00
is_rand分布：
is_rand
0    1414622
Name: count, dtype: int64
随机推荐表is_rand分布：
is_rand
1    1186049
Name: count, dtype: int64


In [42]:
processed_dir = PROCESSED_DIR
processed_dir.mkdir(parents=True, exist_ok=True)
print("清洗数据保存目录：", processed_dir.resolve())


清洗数据保存目录： E:\Users\yanyan\Desktop\KuaiRand_Pure\data\processed


In [43]:
standard_path = (
    processed_dir
    / "log_standard_clean.parquet"
)

random_path = (
    processed_dir
    / "log_random_clean.parquet"
)

df_standard.to_parquet(
    standard_path,
    index=False
)

df.to_parquet(
    random_path,
    index=False
)

print(
    "标准推荐表已导出：",
    standard_path,
    df_standard.shape
)

print(
    "随机推荐表已导出：",
    random_path,
    df.shape
)


标准推荐表已导出： E:\Users\yanyan\Desktop\KuaiRand_Pure\data\processed\log_standard_clean.parquet (1414622, 25)
随机推荐表已导出： E:\Users\yanyan\Desktop\KuaiRand_Pure\data\processed\log_random_clean.parquet (1186049, 25)
